In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import ast
import sys
import numpy as np

from miscellaneous import read_instance

In [7]:
def load_results(filepath='results/optimization_results_all.csv'):
    df = pd.read_csv(filepath)
    df['landing_times'] = df['landing_times'].apply(ast.literal_eval)
    df['runway_assignment'] = df['runway_assignment'].apply(ast.literal_eval)
    # If the file has no 'solver' column (legacy format), default to 'mip'
    if 'solver' not in df.columns:
        df['solver'] = 'mip'
    return df

def get_instance_data(df, instance_number, number_of_runways, objective, solver='mip'):
    row = df[
        (df['solver'] == solver) &
        (df['instance_number'] == instance_number) &
        (df['number_of_runways'] == number_of_runways) &
        (df['objective'] == objective)
    ].iloc[0]
    return row

# Load the data
df = load_results()

# Convenience filtered DataFrames
df_mip = df[df['solver'] == 'mip'].copy()
df_cp  = df[df['solver'] == 'cp'].copy()

instance_numbers = sorted(df['instance_number'].unique())
objectives = ['penalty', 'makespan', 'total_time']
num_runways_list = [1, 2, 3]

merged = df_mip.merge(
    df_cp,
    on=['instance_number', 'number_of_runways', 'objective'],
    suffixes=('_mip', '_cp')
)

dict_obj_names = {'penalty': 'Penalty', 'makespan': 'Makespan', 'total_time': 'Landing Time'}

In [3]:
import os
from miscellaneous import read_instance
from tabulate import tabulate

# List all instance files
instance_folder = 'instances'
instance_files = sorted([f for f in os.listdir(instance_folder) if f.startswith('airland') and f.endswith('.txt')])

summary = []

for fname in instance_files:
    instance_number = int(fname.replace('airland', '').replace('.txt', ''))
    instance = read_instance(instance_number)

    summary.append([
        int(instance_number),
        int(instance['number_of_planes']),
        int(min(instance['appearance_time_vector'])),
        int(max(instance['appearance_time_vector'])),
        int(min(instance['earliest_landing_vector'])),
        int(max(instance['latest_landing_vector'])),
        min(instance['penalty_early_vector']),
        max(instance['penalty_late_vector'])
    ])

headers = [
    'Instance', 'Planes', 'Earliest Appearance', 'Latest Appearance',
    'Earliest Landing', 'Latest Landing', 'Min Early Penalty', 'Max Late Penalty'
]

latex_table = tabulate(sorted(summary), headers, tablefmt='latex_booktabs', floatfmt='.2f')
print(latex_table)

\begin{tabular}{rrrrrrrr}
\toprule
   Instance &   Planes &   Earliest Appearance &   Latest Appearance &   Earliest Landing &   Latest Landing &   Min Early Penalty &   Max Late Penalty \\
\midrule
          1 &       10 &                    14 &                 120 &                 89 &              744 &               10.00 &              30.00 \\
          2 &       15 &                     9 &                 201 &                 84 &              837 &               10.00 &              30.00 \\
          3 &       20 &                     0 &                 235 &                 75 &              967 &               10.00 &              30.00 \\
          4 &       20 &                     7 &                 211 &                 82 &              840 &               10.00 &              30.00 \\
          5 &       20 &                     7 &                 225 &                 82 &              931 &               10.00 &              30.00 \\
          6 &       30 &  

In [4]:
from tabulate import tabulate

for solver_name in ['mip', 'cp']:
    df_s = df[df['solver'] == solver_name]
    if df_s.empty:
        continue

    for obj in objectives:
        print(f"% -- {solver_name.upper()} -- Objective: {obj} --")
        print("\\begin{table}[h!]")
        print("\\centering")
        print("\\small")
        print(f"\\caption{{Results obtained by {solver_name.upper()} when minimizing {obj}.}}")
        print(f"\\label{{tab:{obj}_results_{solver_name}}}")
        print("\\setlength{\\tabcolsep}{6pt}")
        print("\\begin{tabular}{ccrrrrrrr}")
        print("\\toprule")
        print("\\makecell{Instance} & \\makecell{Runway} & \\makecell{Lower\\\\Bound} & \\makecell{Upper\\\\Bound} & \\makecell{Gap\\\\(\\%)} & \\makecell{Time\\\\(s)} & Penalty & Makespan & \\makecell{Landing\\\\Time} \\\\")
        print("\\midrule")

        for inst in instance_numbers:
            for idx, runways in enumerate(num_runways_list):
                row = df_s[(df_s['instance_number'] == inst) &
                           (df_s['number_of_runways'] == runways) &
                           (df_s['objective'] == obj)]
                if not row.empty:
                    row = row.iloc[0]
                    inst_str = f"\\multirow{{3}}{{*}}{{{inst}}}" if idx == 0 else ""
                    print(f"{inst_str} & {runways} & {row['lb']:.2f} & {row['ub']:.2f} & {row['gap']*100:.2f} & {row['time_taken']:.2f} & {row['penalty_objective']:.2f} & {row['makespan_objective']:.0f} & {row['total_time_objective']:.2f} \\\\")
            print("\\midrule")

        print("\\bottomrule")
        print("\\end{tabular}")
        print("\\end{table}")
        print("\n")

% -- MIP -- Objective: penalty --
\begin{table}[h!]
\centering
\small
\caption{Results obtained by MIP when minimizing penalty.}
\label{tab:penalty_results_mip}
\setlength{\tabcolsep}{6pt}
\begin{tabular}{ccrrrrrrr}
\toprule
\makecell{Instance} & \makecell{Runway} & \makecell{Lower\\Bound} & \makecell{Upper\\Bound} & \makecell{Gap\\(\%)} & \makecell{Time\\(s)} & Penalty & Makespan & \makecell{Landing\\Time} \\
\midrule
\multirow{3}{*}{1} & 1 & 700.00 & 700.00 & 0.00 & 0.02 & 700.00 & 258 & 1477.00 \\
 & 2 & 90.00 & 90.00 & 0.00 & 0.06 & 90.00 & 258 & 1484.00 \\
 & 3 & 0.00 & 0.00 & 0.00 & 0.00 & 0.00 & 258 & 1483.00 \\
\midrule
\multirow{3}{*}{2} & 1 & 1480.00 & 1480.00 & 0.00 & 0.07 & 1480.00 & 342 & 2741.00 \\
 & 2 & 210.00 & 210.00 & 0.00 & 0.02 & 210.00 & 342 & 2702.00 \\
 & 3 & 0.00 & 0.00 & 0.00 & 0.01 & 0.00 & 342 & 2695.00 \\
\midrule
\multirow{3}{*}{3} & 1 & 820.00 & 820.00 & 0.00 & 0.02 & 820.00 & 409 & 4094.00 \\
 & 2 & 60.00 & 60.00 & 0.00 & 0.02 & 60.00 & 409 & 4068.00 \\


In [10]:
#Not in the paper
'''
for solver_name in ['mip', 'cp']:
    df_s = df[df['solver'] == solver_name]
    if df_s.empty:
        continue

    for obj in objectives:
        table = []
        for inst in instance_numbers:
            for idx, runways in enumerate(num_runways_list):
                row = df_s[(df_s['instance_number'] == inst) &
                           (df_s['number_of_runways'] == runways) &
                           (df_s['objective'] == obj)]
                if not row.empty:
                    row = row.iloc[0]
                    instance_val = inst if idx == 1 else ""
                    solution_str = (f"Landings: {row['landing_times']}\\n"
                                    f"Runways: {', '.join(str(r) for r in row['runway_assignment'])}")
                    table.append([instance_val, runways, solution_str])

        headers = ["Instance", "Runways", "Solution (Landing Times and Runways)"]
        label = f"tab:solutions_{solver_name}_{obj}"
        caption = (f"Solutions obtained by {solver_name.upper()} "
                   f"when minimizing {dict_obj_names[obj]} "
                   f"for all instances and runway configurations.")

        print(r"\begin{table}[h!]")
        print(r"\centering")
        print(r"\small")
        print(f"\\caption{{{caption}}}")
        print(f"\\label{{{label}}}")
        print(tabulate(table, headers, tablefmt="latex_booktabs"))
        print(r"\end{table}")
        print()
'''

<>:2: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\c'
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_18741/3394163747.py:2: SyntaxWarning: invalid escape sequence '\c'
  '''


'\nfor solver_name in [\'mip\', \'cp\']:\n    df_s = df[df[\'solver\'] == solver_name]\n    if df_s.empty:\n        continue\n\n    for obj in objectives:\n        table = []\n        for inst in instance_numbers:\n            for idx, runways in enumerate(num_runways_list):\n                row = df_s[(df_s[\'instance_number\'] == inst) &\n                           (df_s[\'number_of_runways\'] == runways) &\n                           (df_s[\'objective\'] == obj)]\n                if not row.empty:\n                    row = row.iloc[0]\n                    instance_val = inst if idx == 1 else ""\n                    solution_str = (f"Landings: {row[\'landing_times\']}\\n"\n                                    f"Runways: {\', \'.join(str(r) for r in row[\'runway_assignment\'])}")\n                    table.append([instance_val, runways, solution_str])\n\n        headers = ["Instance", "Runways", "Solution (Landing Times and Runways)"]\n        label = f"tab:solutions_{solver_name}_{ob

In [11]:
for obj in objectives:
    print(f"% -- Comparison: {obj} --")
    print("\\begin{table}[h!]")
    print("\\centering")
    print("\\small")
    print(f"\\caption{{MIP vs CP comparison when minimizing {obj}.}}")
    print(f"\\label{{tab:comparison_{obj}}}")
    print("\\setlength{\\tabcolsep}{4pt}")
    print("\\begin{tabular}{cc rr rr rr}")
    print("\\toprule")
    print(" & & \\multicolumn{2}{c}{Objective Value} & \\multicolumn{2}{c}{Gap (\\%)} & \\multicolumn{2}{c}{Time (s)} \\\\")
    print("\\cmidrule(lr){3-4} \\cmidrule(lr){5-6} \\cmidrule(lr){7-8}")
    print("Instance & RW & MIP & CP & MIP & CP & MIP & CP \\\\")
    print("\\midrule")

    for inst in instance_numbers:
        for idx, runways in enumerate(num_runways_list):
            row_mip = df_mip[(df_mip['instance_number'] == inst) &
                             (df_mip['number_of_runways'] == runways) &
                             (df_mip['objective'] == obj)]
            row_cp = df_cp[(df_cp['instance_number'] == inst) &
                           (df_cp['number_of_runways'] == runways) &
                           (df_cp['objective'] == obj)]

            if row_mip.empty or row_cp.empty:
                continue

            rm = row_mip.iloc[0]
            rc = row_cp.iloc[0]

            if obj == 'penalty':
                val_mip, val_cp = rm['penalty_objective'], rc['penalty_objective']
            elif obj == 'total_time':
                val_mip, val_cp = rm['total_time_objective'], rc['total_time_objective']
            else:
                val_mip, val_cp = rm['makespan_objective'], rc['makespan_objective']

            gap_mip = rm['gap'] * 100
            gap_cp  = rc['gap'] * 100
            t_mip   = rm['time_taken']
            t_cp    = rc['time_taken']

            def bold_min(a, b, fmt=".2f", tol=1e-4):
                sa = f"{a:{fmt}}"
                sb = f"{b:{fmt}}"
                if abs(a - b) < tol:
                    return sa, sb
                if a < b:
                    return f"\\textbf{{{sa}}}", sb
                return sa, f"\\textbf{{{sb}}}"

            sv_mip, sv_cp = bold_min(val_mip, val_cp, fmt=".2f")
            sg_mip, sg_cp = bold_min(gap_mip, gap_cp, fmt=".2f")
            st_mip, st_cp = bold_min(t_mip,   t_cp,   fmt=".2f")

            inst_str = f"\\multirow{{3}}{{*}}{{{inst}}}" if idx == 0 else ""
            print(f"{inst_str} & {runways} & {sv_mip} & {sv_cp} & {sg_mip} & {sg_cp} & {st_mip} & {st_cp} \\\\")

        print("\\midrule")

    print("\\bottomrule")
    print("\\end{tabular}")
    print("\\end{table}")
    print("\n")

% -- Comparison: penalty --
\begin{table}[h!]
\centering
\small
\caption{MIP vs CP comparison when minimizing penalty.}
\label{tab:comparison_penalty}
\setlength{\tabcolsep}{4pt}
\begin{tabular}{cc rr rr rr}
\toprule
 & & \multicolumn{2}{c}{Objective Value} & \multicolumn{2}{c}{Gap (\%)} & \multicolumn{2}{c}{Time (s)} \\
\cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-8}
Instance & RW & MIP & CP & MIP & CP & MIP & CP \\
\midrule
\multirow{3}{*}{1} & 1 & 700.00 & 700.00 & 0.00 & 0.00 & \textbf{0.02} & 0.05 \\
 & 2 & 90.00 & 90.00 & 0.00 & 0.00 & 0.06 & \textbf{0.01} \\
 & 3 & 0.00 & 0.00 & 0.00 & 0.00 & \textbf{0.00} & 0.01 \\
\midrule
\multirow{3}{*}{2} & 1 & 1480.00 & 1480.00 & 0.00 & 0.00 & \textbf{0.07} & 0.10 \\
 & 2 & 210.00 & 210.00 & 0.00 & 0.00 & \textbf{0.02} & 0.02 \\
 & 3 & 0.00 & 0.00 & 0.00 & 0.00 & \textbf{0.01} & 0.02 \\
\midrule
\multirow{3}{*}{3} & 1 & 820.00 & 820.00 & 0.00 & 0.00 & \textbf{0.02} & 0.05 \\
 & 2 & 60.00 & 60.00 & 0.00 & 0.00 & 0.02 & \textbf{0.02

In [12]:
for obj in objectives:
    sub = merged[merged['objective'] == obj].copy()

    if obj == 'penalty':
        sub['val_mip'] = sub['penalty_objective_mip']
        sub['val_cp']  = sub['penalty_objective_cp']
    elif obj == 'total_time':
        sub['val_mip'] = sub['total_time_objective_mip']
        sub['val_cp']  = sub['total_time_objective_cp']
    else:
        sub['val_mip'] = sub['makespan_objective_mip']
        sub['val_cp']  = sub['makespan_objective_cp']

    mip_wins  = (sub['val_mip'] < sub['val_cp'] - 0.01).sum()
    cp_wins   = (sub['val_cp']  < sub['val_mip'] - 0.01).sum()
    ties      = len(sub) - mip_wins - cp_wins

    mip_faster = (sub['time_taken_mip'] < sub['time_taken_cp']).sum()
    cp_faster  = (sub['time_taken_cp']  < sub['time_taken_mip']).sum()

    print(f"-- {obj.upper()} --")
    print(f"  Solution quality:  MIP better={mip_wins}, CP better={cp_wins}, tied={ties}")
    print(f"  Solve time:        MIP faster={mip_faster}, CP faster={cp_faster}")
    print(f"  Avg gap MIP: {sub['gap_mip'].mean()*100:.2f}%   Avg gap CP: {sub['gap_cp'].mean()*100:.2f}%")
    print(f"  Avg time MIP: {sub['time_taken_mip'].mean():.2f}s  Avg time CP: {sub['time_taken_cp'].mean():.2f}s")
    print()

-- PENALTY --
  Solution quality:  MIP better=4, CP better=1, tied=34
  Solve time:        MIP faster=24, CP faster=15
  Avg gap MIP: 6.99%   Avg gap CP: 10.29%
  Avg time MIP: 70.40s  Avg time CP: 60.89s

-- MAKESPAN --
  Solution quality:  MIP better=0, CP better=0, tied=39
  Solve time:        MIP faster=20, CP faster=19
  Avg gap MIP: 0.01%   Avg gap CP: 0.00%
  Avg time MIP: 8.28s  Avg time CP: 0.85s

-- TOTAL_TIME --
  Solution quality:  MIP better=0, CP better=4, tied=35
  Solve time:        MIP faster=15, CP faster=24
  Avg gap MIP: 0.05%   Avg gap CP: 0.03%
  Avg time MIP: 65.86s  Avg time CP: 51.73s



In [ ]:
import numpy as np

GAP_TOL = 1e-6

# Subset where NEITHER solver reached optimality — used for gap stats only
neither_optimal = merged[
    (merged['gap_mip'] > GAP_TOL) & np.isfinite(merged['gap_mip']) &
    (merged['gap_cp']  > GAP_TOL) & np.isfinite(merged['gap_cp'])
][['instance_number', 'number_of_runways', 'objective']]

df_mip_neither = df_mip.merge(neither_optimal, on=['instance_number', 'number_of_runways', 'objective'])
df_cp_neither  = df_cp.merge(neither_optimal,  on=['instance_number', 'number_of_runways', 'objective'])

def n_unsolved(solver_df, obj=None):
    """True count of runs that solver left unsolved, regardless of what the other did."""
    s = solver_df if obj is None else solver_df[solver_df['objective'] == obj]
    return int(((s['gap'] > GAP_TOL) & np.isfinite(s['gap'])).sum())

def gap_stats(solver_df, obj=None):
    """Avg/std gap only over the neither-optimal subset."""
    s = solver_df if obj is None else solver_df[solver_df['objective'] == obj]
    s = s['gap']
    s = s[(s > GAP_TOL) & np.isfinite(s)] * 100
    n = len(s)
    if n == 0:
        return "--", "--"
    return f"{s.mean():.2f}", f"{s.std(ddof=0):.2f}"

print(r"\begin{table}[h!]")
print(r"\centering")
print(r"\small")
print(r"\caption{Per-solver count of configurations not solved to proven optimality "
      r"(\# unsolved) and optimality gap statistics (avg and std) computed over the "
      r"subset of configurations where neither solver reached optimality.}")
print(r"\label{tab:gap_stats}")
print(r"\begin{tabular}{l rr rr rr}")
print(r"\toprule")
print(r" & \multicolumn{3}{c}{MIP} & \multicolumn{3}{c}{CP} \\")
print(r"\cmidrule(lr){2-4} \cmidrule(lr){5-7}")
print(r"Objective & \# unsolved & Avg gap (\%) & Std gap (\%) "
      r"& \# unsolved & Avg gap (\%) & Std gap (\%) \\")
print(r"\midrule")

for obj in objectives:
    mn   = n_unsolved(df_mip, obj)
    cn   = n_unsolved(df_cp,  obj)
    mavg, mstd = gap_stats(df_mip_neither, obj)
    cavg, cstd = gap_stats(df_cp_neither,  obj)
    print(f"{dict_obj_names[obj]} & {mn} & {mavg} & {mstd} & {cn} & {cavg} & {cstd} \\\\")

print(r"\midrule")
mn   = n_unsolved(df_mip)
cn   = n_unsolved(df_cp)
mavg, mstd = gap_stats(df_mip_neither)
cavg, cstd = gap_stats(df_cp_neither)
print(f"\\textbf{{All}} & {mn} & {mavg} & {mstd} & {cn} & {cavg} & {cstd} \\\\")

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

\begin{table}[h!]
\centering
\small
\caption{Per-solver count of configurations not solved to proven optimality (\# unsolved) and optimality gap statistics (avg and std) computed over the subset of configurations where neither solver reached optimality.}
\label{tab:gap_stats}
\begin{tabular}{l rr rr rr}
\toprule
 & \multicolumn{3}{c}{MIP} & \multicolumn{3}{c}{CP} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7}
Objective & \# unsolved & Avg gap (\%) & Std gap (\%) & \# unsolved & Avg gap (\%) & Std gap (\%) \\
\midrule
Penalty & 9 & 34.55 & 12.52 & 7 & 57.32 & 22.58 \\
Makespan & 1 & -- & -- & 0 & -- & -- \\
Landing Time & 8 & 0.30 & 0.21 & 6 & 0.17 & 0.09 \\
\midrule
\textbf{All} & 18 & 18.74 & 19.39 & 13 & 30.94 & 32.96 \\
\bottomrule
\end{tabular}
\end{table}


In [18]:
import numpy as np

TOL = 0.01
GAP_TOL = 1e-6


def fmt_time(mean, std):
    return f"{mean:.2f} $\\pm$ {std:.2f}"

def scoreboard(sub, obj):
    if obj == 'penalty':
        vm, vc = sub['penalty_objective_mip'], sub['penalty_objective_cp']
    elif obj == 'total_time':
        vm, vc = sub['total_time_objective_mip'], sub['total_time_objective_cp']
    else:
        vm, vc = sub['makespan_objective_mip'], sub['makespan_objective_cp']

    mip_better = int((vm < vc - TOL).sum())
    cp_better  = int((vc < vm - TOL).sum())
    ties       = len(sub) - mip_better - cp_better
    mip_faster = int((sub['time_taken_mip'] < sub['time_taken_cp']).sum())
    cp_faster  = int((sub['time_taken_cp']  < sub['time_taken_mip']).sum())

    # Time stats only over configurations where BOTH reached optimality
    both_optimal = sub[
        (sub['gap_mip'] <= GAP_TOL) & np.isfinite(sub['gap_mip']) &
        (sub['gap_cp']  <= GAP_TOL) & np.isfinite(sub['gap_cp'])
    ]
    if both_optimal.empty:
        t_mip = t_cp = "--"
    else:
        t_mip = fmt_time(both_optimal['time_taken_mip'].mean(),
                         both_optimal['time_taken_mip'].std(ddof=0))
        t_cp  = fmt_time(both_optimal['time_taken_cp'].mean(),
                         both_optimal['time_taken_cp'].std(ddof=0))

    return {
        'n': len(sub),
        'mip_better': mip_better, 'cp_better': cp_better, 'ties': ties,
        'mip_faster': mip_faster, 'cp_faster': cp_faster,
        't_mip': t_mip, 't_cp': t_cp,
    }

print(r"\begin{table}[h!]")
print(r"\centering")
print(r"\small")
print(r"\caption{Head-to-head comparison of MIP and CP across matched configurations. "
      r"`Quality wins' counts configurations where a solver found a strictly better "
      r"objective value; `Speed wins' counts configurations where a solver was strictly "
      r"faster. Time is reported as mean $\pm$ std (seconds), computed only over "
      r"configurations where both solvers reached proven optimality.}")
print(r"\label{tab:scoreboard}")
print(r"\begin{tabular}{l c rrr rr rr}")
print(r"\toprule")
print(r" & & \multicolumn{3}{c}{Quality wins} & \multicolumn{2}{c}{Speed wins}"
      r" & \multicolumn{2}{c}{Time (s)} \\")
print(r"\cmidrule(lr){3-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}")
print(r"Objective & \# cfg & MIP & CP & Tie & MIP & CP & MIP & CP \\")
print(r"\midrule")

for obj in objectives:
    s = scoreboard(merged[merged['objective'] == obj], obj)
    print(f"{dict_obj_names[obj]} & {s['n']} "
          f"& {s['mip_better']} & {s['cp_better']} & {s['ties']} "
          f"& {s['mip_faster']} & {s['cp_faster']} "
          f"& {s['t_mip']} & {s['t_cp']} \\\\")

print(r"\midrule")

agg = {'n': 0, 'mip_better': 0, 'cp_better': 0, 'ties': 0,
       'mip_faster': 0, 'cp_faster': 0}
for obj in objectives:
    s = scoreboard(merged[merged['objective'] == obj], obj)
    for k in agg:
        agg[k] += s[k]

both_optimal_all = merged[
    (merged['gap_mip'] <= GAP_TOL) & np.isfinite(merged['gap_mip']) &
    (merged['gap_cp']  <= GAP_TOL) & np.isfinite(merged['gap_cp'])
]
t_mip_all = fmt_time(both_optimal_all['time_taken_mip'].mean(),
                     both_optimal_all['time_taken_mip'].std(ddof=0))
t_cp_all  = fmt_time(both_optimal_all['time_taken_cp'].mean(),
                     both_optimal_all['time_taken_cp'].std(ddof=0))

print(f"\\textbf{{All}} & {agg['n']} "
      f"& {agg['mip_better']} & {agg['cp_better']} & {agg['ties']} "
      f"& {agg['mip_faster']} & {agg['cp_faster']} "
      f"& {t_mip_all} & {t_cp_all} \\\\")

print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

\begin{table}[h!]
\centering
\small
\caption{Head-to-head comparison of MIP and CP across matched configurations. `Quality wins' counts configurations where a solver found a strictly better objective value; `Speed wins' counts configurations where a solver was strictly faster. Time is reported as mean $\pm$ std (seconds), computed only over configurations where both solvers reached proven optimality.}
\label{tab:scoreboard}
\begin{tabular}{l c rrr rr rr}
\toprule
 & & \multicolumn{3}{c}{Quality wins} & \multicolumn{2}{c}{Speed wins} & \multicolumn{2}{c}{Time (s)} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}
Objective & \# cfg & MIP & CP & Tie & MIP & CP & MIP & CP \\
\midrule
Penalty & 39 & 4 & 1 & 34 & 24 & 15 & 1.05 $\pm$ 3.71 & 2.32 $\pm$ 5.98 \\
Makespan & 39 & 0 & 0 & 39 & 20 & 19 & 0.56 $\pm$ 1.18 & 0.21 $\pm$ 0.31 \\
Landing Time & 39 & 0 & 4 & 35 & 15 & 24 & 5.16 $\pm$ 17.84 & 4.90 $\pm$ 20.00 \\
\midrule
\textbf{All} & 117 & 4 & 5 & 108 & 59 & 58 & 2.15 $\pm$ 10

In [11]:
for idx, row in merged.iterrows():
    obj = row['objective']          # <-- the missing line; use the row's own objective

    if obj == 'penalty':
        val_mip = row['penalty_objective_mip']
        val_cp  = row['penalty_objective_cp']
    elif obj == 'total_time':
        val_mip = row['total_time_objective_mip']
        val_cp  = row['total_time_objective_cp']
    elif obj == 'makespan':
        val_mip = row['makespan_objective_mip']
        val_cp  = row['makespan_objective_cp']

    if abs(val_mip - val_cp) > 0.01:
        print(f"Instance {row['instance_number']}, Runways {row['number_of_runways']}, Objective {obj}:")
        print(f"  MIP: Value={val_mip:.2f}, MIP Gap={row['gap_mip']*100:.2f}%, Time={row['time_taken_mip']:.2f}s")
        print(f"  CP:  Value={val_cp:.2f}, CP Gap={row['gap_cp']*100:.2f}%, Time={row['time_taken_cp']:.2f}s")  
        print()

Instance 9, Runways 1, Objective total_time:
  MIP: Value=607095.00, MIP Gap=0.58%, Time=300.10s
  CP:  Value=607054.00, CP Gap=0.23%, Time=303.17s

Instance 10, Runways 1, Objective penalty:
  MIP: Value=13154.81, MIP Gap=53.67%, Time=300.18s
  CP:  Value=12944.17, CP Gap=68.12%, Time=300.80s

Instance 10, Runways 1, Objective total_time:
  MIP: Value=1419715.00, MIP Gap=0.59%, Time=300.29s
  CP:  Value=1419615.00, CP Gap=0.28%, Time=302.77s

Instance 11, Runways 1, Objective penalty:
  MIP: Value=12418.32, MIP Gap=32.30%, Time=300.30s
  CP:  Value=12592.89, CP Gap=72.94%, Time=301.39s

Instance 12, Runways 1, Objective penalty:
  MIP: Value=16382.18, MIP Gap=40.76%, Time=300.47s
  CP:  Value=16532.52, CP Gap=56.18%, Time=301.33s

Instance 13, Runways 1, Objective penalty:
  MIP: Value=37605.57, MIP Gap=48.03%, Time=301.71s
  CP:  Value=40613.82, CP Gap=78.71%, Time=301.37s

Instance 13, Runways 1, Objective total_time:
  MIP: Value=14370270.00, MIP Gap=0.18%, Time=301.86s
  CP:  Valu